In [1]:
# NoorSuite / SciSuite - exploratory (Jupyter) side.
# Install once from the repo root:  pip install -e .
import numpy as np
import pandas as pd

from NoorSuite import SciSuiteClient

# Attach to a running SciSuite window, or start one as a background process.
suite = SciSuiteClient().launch()
suite

[launch] Started SciSuite process on port 55555.


<SciSuiteClient port=55555 offline>

In [ ]:
# Re-run this after restarting the GUI process to reconnect.
suite.launch()

In [3]:
np.random.rand(100,100,100)

array([[[0.45709861, 0.3767779 , 0.32527039, ..., 0.30845577,
         0.95655675, 0.41600799],
        [0.62338499, 0.61014772, 0.55135042, ..., 0.8313494 ,
         0.24647503, 0.93142404],
        [0.97507897, 0.57442403, 0.41607158, ..., 0.63200903,
         0.62384698, 0.71969449],
        ...,
        [0.82929081, 0.06614794, 0.91951969, ..., 0.01195668,
         0.95622508, 0.41611626],
        [0.83575104, 0.51849593, 0.57546792, ..., 0.83090791,
         0.59611807, 0.64559048],
        [0.63105849, 0.98201084, 0.89402491, ..., 0.82274836,
         0.25265675, 0.66764852]],

       [[0.49552213, 0.04709425, 0.74310347, ..., 0.27292152,
         0.68371479, 0.51715809],
        [0.15399892, 0.53849313, 0.58522088, ..., 0.76123141,
         0.35932385, 0.47859155],
        [0.74275324, 0.37228074, 0.69953986, ..., 0.64643269,
         0.50729957, 0.66892978],
        ...,
        [0.32105014, 0.23705021, 0.53954789, ..., 0.85234939,
         0.74549608, 0.16289956],
        [0.1

In [4]:
suite.push_image(np.random.rand(100,100,100), name="scan", axis_names=["z", "y", "x"], tags=["imaging"])

{'status': 'success'}

In [8]:
suite.list_data()

,id,name,columns,nrows,tags,source
0,1a70c847,Experiment_Run_1,"[Time_s, Signal_A, Signal_B]",100,"[demo, run_1]",dataframe


In [7]:
# Push a DataFrame -> it becomes ONE data object in the pool (its columns are
# pointers). Nothing is plotted yet.
df = pd.DataFrame({
    "Time_s": np.linspace(0, 10, 100),
    "Signal_A": np.sin(np.linspace(0, 10, 100)),
    "Signal_B": np.cos(np.linspace(0, 10, 100)) + np.random.normal(0, 0.1, 100),
})
suite.push_dataframe(df, name="Experiment_Run_1", tags=["demo", "run_1"])

# Then choose x + y column(s) and send them to a sheet. (In the GUI: tick X/Y in
# the column panel and click "Add to active subplot", or drag onto a sheet.)
suite.plot("Experiment_Run_1", x="Time_s", y=["Signal_A", "Signal_B"], new_sheet=True)

{'status': 'success'}

In [ ]:
# push + plot in one call: pass a DataFrame straight to .plot().
df2 = pd.DataFrame({
    "time_s": np.linspace(0, 10, 200),
    "current_mA": np.sin(np.linspace(0, 10, 200)) * 10,
    "voltage_V": np.cos(np.linspace(0, 10, 200)) * 5,
})
suite.plot(df2, x="time_s", y=["current_mA", "voltage_V"],
           name="IV_sweep", new_sheet=True)

In [ ]:
# What is in the pool, and what is plotted on the sheets.
display(suite.list_data())
suite.list_traces()

In [ ]:
# mode="update" refreshes the data object in place (same name), so every trace
# already pointing at "IV_sweep" re-renders with the new numbers.
df2["current_mA"] = np.sin(np.linspace(0, 10, 200)) * 12 + 1.5
suite.push_dataframe(df2, name="IV_sweep", mode="update")

In [ ]:
# Pull a data object back, edit it, push it -> the GUI stays in sync.
df3 = suite.get_data("IV_sweep")          # or get_dataframe(...)
df3["power_mW"] = df3["current_mA"] * df3["voltage_V"]
suite.push_dataframe(df3, name="IV_sweep", mode="update")   # new column appears in the GUI

In [ ]:
# ND image data: push an array, then show a 2-D slice with a scroll slider.
stack = np.random.default_rng(0).random((30, 128, 160)).astype("float32")   # (z, y, x)
for z in range(30):                                                          # a moving blob
    yy, xx = np.mgrid[0:128, 0:160]
    stack[z] += np.exp(-(((xx - 20 - 4 * z) ** 2 + (yy - 64) ** 2) / 400.0))

suite.push_image(stack, name="z_stack", axis_names=["z", "y", "x"], tags=["imaging"])
suite.show_image("z_stack", axes=(1, 2), new_sheet=True)     # display y-x; slider scrolls z

# In the GUI the slider bar has Row (y) / Col (x) / Slice dropdowns - change them to
# scroll along x or y instead. list_images() / get_image() round-trip the array:
# arr = suite.get_image("z_stack")

In [ ]:
# Housekeeping.
# suite.remove_data("IV_sweep")   # drop a data object + every trace using it
# suite.clear()                   # wipe the whole pool and all sheets

---
**Everything else is GUI-side.** Colours & colormaps (per-sheet, with a project library
and `.scicmap` export/import), plot tags + fuzzy search, grid styling, subplot reordering,
per-sheet notes, image display settings (cmap / vmin-vmax / colourbar), and multi-select
bulk trace edits (`*2` / `+0.5` style expressions) all live in the NOORSUITE window.
Projects save as `<name>.sciproj` with a `<name>/` sidecar folder holding image `.npy` files;
**Ctrl+S** saves and the project autosaves every ~2 min once a filename is set.
